# generate_masks

This copies split-wise pre/post images into `data/processed`, rasterizes GeoJSON annotations into mask PNGs, and updates `metadata.csv`.

## Imports and project paths

In [1]:
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
from PIL import Image

import rasterio
from rasterio.features import rasterize
from shapely.geometry import shape

def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()

    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "src").exists():
            return p

    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p

    return start

PROJECT_ROOT = find_project_root()
METADATA_PATH = PROJECT_ROOT / "data" / "processed" / "metadata.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("METADATA_PATH:", METADATA_PATH)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project
METADATA_PATH: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project\data\processed\metadata.csv
PROCESSED_DIR: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project\data\processed


## Path helpers and processed folder creation

In [2]:
def normalize_rel_path(p):
    p = Path(str(p).replace("\\", "/"))
    parts = list(p.parts)
    if parts and parts[0] == "src":
        p = Path(*parts[1:])
    return p

def to_abs_path(p):
    p = normalize_rel_path(p)
    if p.is_absolute():
        return p
    return (PROJECT_ROOT / p).resolve()

def to_rel_str(p):
    p = normalize_rel_path(p)
    if p.is_absolute():
        try:
            return str(p.resolve().relative_to(PROJECT_ROOT.resolve())).replace("\\", "/")
        except Exception:
            return str(p).replace("\\", "/")
    return str(p).replace("\\", "/")

for split in ["train", "val", "test"]:
    (PROCESSED_DIR / split / "pre").mkdir(parents=True, exist_ok=True)
    (PROCESSED_DIR / split / "post").mkdir(parents=True, exist_ok=True)
    (PROCESSED_DIR / split / "mask").mkdir(parents=True, exist_ok=True)

print("Created/verified processed pre, post, mask folders.")


Created/verified processed pre, post, mask folders.


## Mask generation helper functions

In [5]:
def rasterize_geojson_to_mask(label_path, ref_image_path):
    label_path = to_abs_path(label_path)
    ref_image_path = to_abs_path(ref_image_path)

    with rasterio.open(ref_image_path) as src:
        out_shape = (src.height, src.width)
        transform = src.transform

    with open(label_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    shapes = []
    for feat in data.get("features", []):
        geom = feat.get("geometry")
        if geom is None:
            continue
        try:
            shapes.append((shape(geom), 1))
        except Exception:
            continue

    if not shapes:
        return np.zeros(out_shape, dtype=np.uint8)

    return rasterize(
        shapes=shapes,
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype="uint8"
    )

def save_mask_png(mask, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    mask_img = (mask * 255).astype(np.uint8)
    Image.fromarray(mask_img).save(out_path)

def copy_image(src_path, dst_path):
    src_path = to_abs_path(src_path)
    dst_path = Path(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_path, dst_path)


## Run mask generation and update metadata

In [4]:
df = pd.read_csv(METADATA_PATH)

required_cols = {"sample_id", "split", "pre_path", "post_path", "label_path"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"metadata.csv is missing required columns: {missing}")

for col in ["pre_path", "post_path", "label_path"]:
    df[col] = df[col].apply(to_rel_str)

new_pre_paths = []
new_post_paths = []
new_mask_paths = []
statuses = []

print("Loaded rows:", len(df))

for row in df.itertuples(index=False):
    sample_id = row.sample_id
    split = row.split
    pre_path = row.pre_path
    post_path = row.post_path
    label_path = row.label_path

    try:
        pre_ext = to_abs_path(pre_path).suffix
        post_ext = to_abs_path(post_path).suffix

        out_pre_path = PROCESSED_DIR / split / "pre" / f"{sample_id}{pre_ext}"
        out_post_path = PROCESSED_DIR / split / "post" / f"{sample_id}{post_ext}"
        out_mask_path = PROCESSED_DIR / split / "mask" / f"{sample_id}.png"

        copy_image(pre_path, out_pre_path)
        copy_image(post_path, out_post_path)

        mask = rasterize_geojson_to_mask(label_path, post_path)
        save_mask_png(mask, out_mask_path)

        new_pre_paths.append(to_rel_str(out_pre_path))
        new_post_paths.append(to_rel_str(out_post_path))
        new_mask_paths.append(to_rel_str(out_mask_path))
        statuses.append("OK")

    except Exception as e:
        new_pre_paths.append(None)
        new_post_paths.append(None)
        new_mask_paths.append(None)
        statuses.append(f"ERROR: {e}")
        print(f"Failed for {sample_id}: {e}")

df["pre_path"] = new_pre_paths
df["post_path"] = new_post_paths
df["mask_path"] = new_mask_paths
df["mask_status"] = statuses

df.to_csv(METADATA_PATH, index=False)

print("\nProcessing complete.")
print("Successful rows:", (df["mask_status"] == "OK").sum())
print("Failures:", (df["mask_status"] != "OK").sum())
print("Updated metadata saved to:", METADATA_PATH)
display(df[["sample_id", "split", "pre_path", "post_path", "label_path", "mask_path", "mask_status"]].head())


Loaded rows: 202

Processing complete.
Successful rows: 202
Failures: 0
Updated metadata saved to: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project\data\processed\metadata.csv


,sample_id,split,pre_path,post_path,label_path,mask_path,mask_status
0,sample_00097,train,data/processed/train/pre/sample_00097.tif,data/processed/train/post/sample_00097.tif,data/raw/annotations/0_28_62.geojson,data/processed/train/mask/sample_00097.png,OK
1,sample_00031,train,data/processed/train/pre/sample_00031.tif,data/processed/train/post/sample_00031.tif,data/raw/annotations/0_39_67.geojson,data/processed/train/mask/sample_00031.png,OK
2,sample_00012,train,data/processed/train/pre/sample_00012.tif,data/processed/train/post/sample_00012.tif,data/raw/annotations/0_17_66.geojson,data/processed/train/mask/sample_00012.png,OK
3,sample_00035,train,data/processed/train/pre/sample_00035.tif,data/processed/train/post/sample_00035.tif,data/raw/annotations/0_25_70.geojson,data/processed/train/mask/sample_00035.png,OK
4,sample_00119,train,data/processed/train/pre/sample_00119.tif,data/processed/train/post/sample_00119.tif,data/raw/annotations/0_37_69.geojson,data/processed/train/mask/sample_00119.png,OK
